# SafeVision Notebook 4: Rule search with hybrid retrieval (N4, RAG)

**Goal:** for any safety problem SafeVision finds ("*worker without a hard hat under a crane*"), find the **exact regulation paragraphs** that apply, so the report writer (Notebook 5) can quote real rules instead of inventing them.

**Rule book:** US OSHA construction standard **29 CFR 1926**, downloaded section by section from the official eCFR API (public domain). Nine subparts: C general safety, D health (noise, lighting, first aid), E PPE, F fire, G signs and flaggers, L scaffolds, M fall protection, P excavations, X ladders and stairs. **Optional:** Dubai/UAE rule PDFs can be uploaded and are indexed the same way.

**Pipeline (the design follows [10] and [9]):**
1. Split each section into small **chunks** (each chunk keeps its section number and title).
2. **Keyword search** (BM25) and **meaning search** (bge-small embeddings + FAISS).
3. **Hybrid:** combine both ranked lists with **reciprocal rank fusion (RRF)**.
4. **Re-ranker:** a cross-encoder reads question and chunk together and re-orders the top 20.

| ID | Question | Compared | Scores |
|---|---|---|---|
| **E6a** | Which search method finds the right rule? | BM25 / dense / hybrid (RRF) / hybrid + re-ranker | Recall@1, @3, @5, MRR, ms per query |
| **E6b** | Which chunk size works best? | 100 / 200 / 400 words | same |
| **E6c** *(optional)* | Does a different embedding model help? | bge-small vs all-MiniLM-L6 | same |

**Test questions:** 32 built-in questions written as **site observations** (how SafeVision will actually ask), each with the correct section number(s). Extra questions can be added from a CSV file. Paper [10] reports a jump from 53.7% to 75.7% retrieval accuracy with hybrid search, and paper [9] reports F1 87.3% vs 75.0% against meaning-only search; E6a tests whether we see the same pattern.

**Runtime:** GPU recommended (re-ranker), about 10-15 min.

## 0. Install libraries

In [ ]:
# [colab-only]
%pip install -q sentence-transformers faiss-cpu rank-bm25 pypdf openpyxl huggingface_hub

## 1. Configuration (edit this cell only)

In [ ]:
# CONFIG
from pathlib import Path
import re, json, time

OUT = Path("/content/safevision_outputs_nb4")
ECFR_API = "https://www.ecfr.gov/api/versioner/v1"
SUBPARTS = {"C": "General safety and health", "D": "Occupational health (noise, light, first aid)",
            "E": "Personal protective equipment", "F": "Fire protection", "G": "Signs, signals, barricades",
            "L": "Scaffolds", "M": "Fall protection", "P": "Excavations", "X": "Stairways and ladders"}
ECFR_LOCAL_DIR = None            # folder with already-downloaded subpart XML files (used if the API is down)
EXTRA_PDF_DIR = Path("/content/extra_rules")   # put Dubai/UAE rule PDFs here (optional)
TEAM_QUESTIONS_CSV = None        # optional CSV with columns: question, gold_sections (e.g. "1926.100;1926.95")

CHUNK_WORDS = 200                # main chunk size (E6a); E6b compares the sizes below
CHUNK_SIZES = [100, 200, 400]
CHUNK_OVERLAP = 40               # words shared by neighbouring chunks
EMB_MODEL = "BAAI/bge-small-en-v1.5"
EMB_MODEL_ALT = "sentence-transformers/all-MiniLM-L6-v2"   # E6c
RUN_E6C = True
RERANKER = "BAAI/bge-reranker-base"
RRF_K = 60                       # constant from the original RRF paper (Cormack et al., 2009)
CANDIDATES = 20                  # how many hybrid results the re-ranker re-orders

RES, FIGS, MODELS = OUT / "results", OUT / "figures", OUT / "models"
for p in [OUT, RES, FIGS, MODELS, EXTRA_PDF_DIR]:
    p.mkdir(parents=True, exist_ok=True)

## 2. Results logger
Same format as the other notebooks.

In [ ]:
# Results logger
import csv, datetime
import pandas as pd
LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "method", "chunk_words", "emb_model", "dataset", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, method, chunk_words, emb_model, metric, value, notes="", dataset="OSHA 1926 questions"):
    # Append one measured number to the results log.
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, method=method, chunk_words=chunk_words, emb_model=emb_model,
                        dataset=dataset, metric=metric, value=round(float(value), 4), notes=notes,
                        timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Download 29 CFR 1926 from the eCFR API

The API is free and needs no key. We first ask for the latest issue date of Title 29, then download each subpart as XML. Every `<DIV8 TYPE="SECTION">` element is one regulation section (for example **§ 1926.100 Head protection**).

In [ ]:
# Download 29 CFR 1926 from the eCFR API
import requests, xml.etree.ElementTree as ET

def get_subpart_xml(sub):
    # Return the XML text of one subpart of 29 CFR 1926 (from the local folder or the eCFR API).
    if ECFR_LOCAL_DIR:
        return (Path(ECFR_LOCAL_DIR) / f"1926_subpart_{sub}.xml").read_text()
    date = next(t["latest_issue_date"] for t in requests.get(f"{ECFR_API}/titles.json", timeout=60).json()["titles"] if t["number"] == 29)
    r = requests.get(f"{ECFR_API}/full/{date}/title-29.xml", params={"part": "1926", "subpart": sub}, timeout=120)
    r.raise_for_status()
    return r.text

def text_of(el):
    # All text inside an XML element, with spaces between pieces.
    return re.sub(r"\s+", " ", " ".join(el.itertext())).strip()

sections = []
for sub, sub_title in SUBPARTS.items():
    root = ET.fromstring(get_subpart_xml(sub))
    for div in root.iter("DIV8"):
        if div.get("TYPE") != "SECTION":
            continue
        head = div.find("HEAD")
        heading = text_of(head) if head is not None else div.get("N")
        # keep paragraphs, tables and notes; drop the title and the legal citation line
        paras = [text_of(p) for p in div if p.tag not in ("HEAD", "CITA", "SECAUTH", "AUTH", "SOURCE", "EDNOTE")]
        body = " ".join(p for p in paras if p)
        if len(body.split()) < 15:                 # skip '[Reserved]' sections
            continue
        sections.append({"source": "OSHA 29 CFR 1926", "subpart": sub, "subpart_title": sub_title,
                         "section": div.get("N"), "heading": heading.replace("§", "").strip(), "text": body})
    print(f"Subpart {sub}: {sum(s['subpart'] == sub for s in sections)} sections")
sections = pd.DataFrame(sections)
print("Total sections:", len(sections), "| words:", sections.text.str.split().str.len().sum())

## 4. (Optional) Add Dubai / UAE rule PDFs

Upload PDFs into `EXTRA_PDF_DIR` (for example the Dubai Municipality construction safety code). Each page becomes one "section" named after the file and page, so it can be quoted in the same way. If the folder is empty, this step does nothing.

In [ ]:
# [colab-only]
from google.colab import files
print("Optional: upload Dubai/UAE rule PDFs (cancel the dialog to skip).")
try:
    for name, data in files.upload().items():
        (EXTRA_PDF_DIR / name).write_bytes(data)
except Exception as e:
    print("Skipped:", e)

In [ ]:
# (Optional) Add Dubai / UAE rule PDFs
from pypdf import PdfReader
extra = []
for pdf in sorted(EXTRA_PDF_DIR.glob("*.pdf")):
    for i, page in enumerate(PdfReader(pdf).pages, start=1):
        t = re.sub(r"\s+", " ", page.extract_text() or "").strip()
        if len(t.split()) >= 30:
            extra.append({"source": pdf.stem, "subpart": "-", "subpart_title": "UAE/Dubai document",
                          "section": f"{pdf.stem} p.{i}", "heading": f"{pdf.stem}, page {i}", "text": t})
if extra:
    sections = pd.concat([sections, pd.DataFrame(extra)], ignore_index=True)
print("Extra PDF pages added:", len(extra), "| total sections:", len(sections))

## 5. Chunking

Long sections are cut into overlapping word windows. Every chunk starts with its section number, title and subpart (e.g. "*1926.451 General requirements (Scaffolds).*"), so both search methods can see which rule it belongs to, and the report can cite it.

In [ ]:
# Chunking
def make_chunks(secs, size, overlap=CHUNK_OVERLAP):
    # Split every section into chunks of about `size` words; each chunk keeps its section metadata.
    rows = []
    for _, s in secs.iterrows():
        words = s.text.split()
        step = max(size - overlap, 1)
        for start in range(0, max(len(words) - overlap, 1), step):
            piece = " ".join(words[start:start + size])
            rows.append({**s.drop("text").to_dict(), "chunk_id": f"{s.section}#{start // step}",
                         "text": f"{s.heading} ({s.subpart_title}). {piece}"})
    return pd.DataFrame(rows)

chunk_sets = {n: make_chunks(sections, n) for n in CHUNK_SIZES}
for n, c in chunk_sets.items():
    print(f"{n:4d} words per chunk -> {len(c):5d} chunks")
print(chunk_sets[CHUNK_WORDS].text.iloc[0][:400])

## 6. Test questions (written as site observations)

Each question lists the section(s) that a safety officer would cite. A search result counts as **correct** if its chunk comes from one of these sections. If an extra question CSV is set in the configuration, its questions are added and scored separately.

In [ ]:
# Test questions (written as site observations)
QUESTIONS = [
    ("Worker on site is not wearing a hard hat while loads are moving overhead.", ["1926.100"]),
    ("Who has to provide and pay for the workers' protective equipment?", ["1926.95"]),
    ("Worker is using a jackhammer all day without any ear protection.", ["1926.101", "1926.52"]),
    ("Welder is cutting steel without eye or face protection.", ["1926.102"]),
    ("Worker is cutting concrete in heavy dust without a respirator.", ["1926.103"]),
    ("Worker is tied off with an old body belt and lanyard.", ["1926.104", "1926.502"]),
    ("Workers are 30 feet above the ground with no safety net or other protection.", ["1926.105", "1926.501"]),
    ("Worker on a pier over a river is not wearing a life jacket.", ["1926.106"]),
    ("Worker is wearing sandals instead of safety boots.", ["1926.96"]),
    ("Unprotected slab edge 8 feet above the lower level with no guardrail.", ["1926.501"]),
    ("How high must the top rail of a guardrail system be?", ["1926.502"]),
    ("Has the worker been trained to recognize fall hazards?", ["1926.503"]),
    ("Scaffold platform is not fully planked and has gaps between boards.", ["1926.451"]),
    ("Scaffold was erected without a competent person supervising.", ["1926.451"]),
    ("Workers on the scaffold have not received scaffold training.", ["1926.454"]),
    ("Portable ladder does not extend above the upper landing.", ["1926.1053"]),
    ("Worker is standing on the top step of a stepladder.", ["1926.1053"]),
    ("There is no stairway or ladder at a 19-inch change in elevation.", ["1926.1051"]),
    ("Workers have not been trained on ladder and stairway hazards.", ["1926.1060"]),
    ("Trench is 6 feet deep with no shoring, shielding or sloping.", ["1926.652"]),
    ("There is no ladder to get out of a 5 foot deep trench.", ["1926.651"]),
    ("Excavated soil is piled right at the edge of the excavation.", ["1926.651"]),
    ("There is no first aid kit or person trained in first aid on site.", ["1926.50"]),
    ("No fire extinguisher is available near the welding area.", ["1926.150"]),
    ("Danger zone has no warning signs or barricades.", ["1926.200"]),
    ("Flagger is directing traffic without a high-visibility garment.", ["1926.201"]),
    ("The employer has no safety program and no inspections by a competent person.", ["1926.20"]),
    ("Workers were never instructed to recognize unsafe conditions.", ["1926.21"]),
    ("A floor hole on the second level is left uncovered.", ["1926.501", "1926.502"]),
    ("Scrap lumber with protruding nails is left in walkways.", ["1926.25"]),
    ("Workers are exposed to noise above permissible levels.", ["1926.52"]),
    ("The work area at night is poorly lit.", ["1926.56"]),
]
qa = pd.DataFrame(QUESTIONS, columns=["question", "gold"]).assign(set="built-in")
if TEAM_QUESTIONS_CSV and Path(TEAM_QUESTIONS_CSV).exists():
    team = pd.read_csv(TEAM_QUESTIONS_CSV)
    team["gold"] = team.gold_sections.astype(str).str.split(r"\s*;\s*")
    qa = pd.concat([qa, team[["question", "gold"]].assign(set="team")], ignore_index=True)
known = set(sections.section)
missing = sorted({g for gs in qa.gold for g in gs} - known)
print(len(qa), "questions | gold sections not found in the rule book:", missing or "none")

## 7. Search methods (BM25, dense, hybrid RRF, hybrid + re-ranker)

All four methods share one interface: `search(question, k)` returns chunk row numbers, best first.
- **RRF** score of a chunk = Σ 1 / (60 + rank) over the two lists. It needs no score scaling, which is why [10] and many RAG systems use it.
- The **re-ranker** reads (question, chunk) pairs together, which is slower but more precise, so it only re-orders the top `CANDIDATES` hybrid results.

In [ ]:
# Search methods (BM25, dense, hybrid RRF, hybrid + re-ranker)
import numpy as np, torch, faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STOP = set("a an the of to in on for and or with without is are was were be by at from this that it as any all no not".split())
tokenize = lambda t: [w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOP]
reranker = CrossEncoder(RERANKER, device=DEVICE)
_embedders = {}

def get_embedder(name):
    if name not in _embedders:
        _embedders[name] = SentenceTransformer(name, device=DEVICE)
    return _embedders[name]

class RuleIndex:
    # BM25 + FAISS index over one chunk set, with the four search methods.
    def __init__(self, chunks, emb_name):
        self.chunks = chunks.reset_index(drop=True)
        self.bm25 = BM25Okapi([tokenize(t) for t in self.chunks.text])
        self.emb = get_embedder(emb_name)
        # bge models expect this instruction in front of short search queries
        self.prefix = "Represent this sentence for searching relevant passages: " if "bge" in emb_name else ""
        vecs = self.emb.encode(list(self.chunks.text), batch_size=64, normalize_embeddings=True).astype("float32")
        self.faiss = faiss.IndexFlatIP(vecs.shape[1]); self.faiss.add(vecs)

    def bm25_search(self, q, k):
        return list(np.argsort(-self.bm25.get_scores(tokenize(q)))[:k])

    def dense_search(self, q, k):
        v = self.emb.encode([self.prefix + q], normalize_embeddings=True).astype("float32")
        return list(self.faiss.search(v, k)[1][0])

    def hybrid_search(self, q, k):
        scores = {}
        for ranked in (self.bm25_search(q, CANDIDATES), self.dense_search(q, CANDIDATES)):
            for rank, idx in enumerate(ranked):
                scores[idx] = scores.get(idx, 0) + 1 / (RRF_K + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)[:k]

    def rerank_search(self, q, k):
        cand = self.hybrid_search(q, CANDIDATES)
        s = reranker.predict([(q, self.chunks.text[i]) for i in cand], batch_size=32)
        return [cand[i] for i in np.argsort(-np.asarray(s))][:k]

    def search(self, method, q, k=10):
        return {"BM25": self.bm25_search, "dense": self.dense_search,
                "hybrid (RRF)": self.hybrid_search, "hybrid + re-ranker": self.rerank_search}[method](q, k)

METHODS = ["BM25", "dense", "hybrid (RRF)", "hybrid + re-ranker"]

## 8. Evaluation helper

- **Recall@k:** share of questions where a correct section appears in the top k results (k = 1, 3, 5).
- **MRR:** mean of 1 / (position of the first correct result), 0 if none in the top 10.
- **ms/query:** average search time, because the app has to answer quickly.

In [ ]:
# Evaluation helper
def evaluate(index, method, questions):
    # Return a dict of Recall@1/3/5, MRR@10 and milliseconds per query for one method.
    hits_at = {1: [], 3: [], 5: []}; rr = []; t0 = time.time()
    for _, q in questions.iterrows():
        ranked = index.search(method, q.question, k=10)
        ok = [index.chunks.section[i] in q.gold for i in ranked]
        for k in hits_at:
            hits_at[k].append(any(ok[:k]))
        rr.append(1 / (ok.index(True) + 1) if True in ok else 0)
    return {"Recall@1": np.mean(hits_at[1]), "Recall@3": np.mean(hits_at[3]), "Recall@5": np.mean(hits_at[5]),
            "MRR@10": np.mean(rr), "ms_per_query": 1000 * (time.time() - t0) / len(questions)}

def run_and_log(exp, index, chunk_words, emb_name, methods, questions=None, qset="built-in"):
    questions = qa[qa.set == qset] if questions is None else questions
    rows = []
    for m in methods:
        r = evaluate(index, m, questions)
        for metric, v in r.items():
            log_result(exp, f"{exp}_{m}_{chunk_words}w", m, chunk_words, emb_name, metric, v, dataset=f"{qset} questions")
        rows.append({"experiment": exp, "method": m, "chunk_words": chunk_words, "emb_model": emb_name.split("/")[-1], **r})
    return pd.DataFrame(rows)

## 9. Experiment E6a: Which search method finds the right rule?
Chunk size 200 words, bge-small embeddings.

In [ ]:
# Experiment E6a: Which search method finds the right rule?
main_index = RuleIndex(chunk_sets[CHUNK_WORDS], EMB_MODEL)
e6a = run_and_log("E6a", main_index, CHUNK_WORDS, EMB_MODEL, METHODS)
if (qa.set == "team").any():
    e6a = pd.concat([e6a, run_and_log("E6a", main_index, CHUNK_WORDS, EMB_MODEL, METHODS, qset="team").assign(experiment="E6a (team questions)")])
print(e6a.round(3).to_string(index=False))

## 10. Experiment E6b: Which chunk size works best?
Small chunks are precise but can lose context; large chunks keep context but mix topics. Tested with the two hybrid methods.

In [ ]:
# Experiment E6b: Which chunk size works best?
e6b = pd.concat([run_and_log("E6b", RuleIndex(chunk_sets[n], EMB_MODEL), n, EMB_MODEL, ["hybrid (RRF)", "hybrid + re-ranker"])
                 for n in CHUNK_SIZES])
print(e6b.round(3).to_string(index=False))

## 11. Experiment E6c (optional): A different embedding model

In [ ]:
# Experiment E6c (optional): A different embedding model
e6c = pd.DataFrame()
if RUN_E6C:
    e6c = run_and_log("E6c", RuleIndex(chunk_sets[CHUNK_WORDS], EMB_MODEL_ALT), CHUNK_WORDS, EMB_MODEL_ALT, ["dense", "hybrid (RRF)", "hybrid + re-ranker"])
    print(e6c.round(3).to_string(index=False))

## 12. Retrieval chart

In [ ]:
# Retrieval chart
import matplotlib.pyplot as plt
base = e6a[e6a.experiment == "E6a"].set_index("method")[["Recall@1", "Recall@3", "Recall@5", "MRR@10"]]
ax = base.plot(kind="bar", figsize=(9, 4), rot=0)
ax.set_ylim(0, 1); ax.set_ylabel("score (built-in questions)"); ax.set_title("E6a: rule search method")
plt.tight_layout(); plt.savefig(FIGS / "fig_E6a_methods.png", dpi=150); plt.show()

piv = e6b.pivot_table(index="chunk_words", columns="method", values="Recall@3")
ax = piv.plot(marker="o", figsize=(6, 4)); ax.set_ylim(0, 1); ax.set_ylabel("Recall@3"); ax.set_title("E6b: chunk size")
plt.tight_layout(); plt.savefig(FIGS / "fig_E6b_chunk_size.png", dpi=150); plt.show()

## 13. Failure check: questions the best method still misses
Useful for the error-analysis section of the report.

In [ ]:
# Failure check: questions the best method still misses
misses = []
for _, q in qa.iterrows():
    top = main_index.search("hybrid + re-ranker", q.question, k=3)
    got = [main_index.chunks.section[i] for i in top]
    if not set(got) & set(q.gold):
        misses.append({"question": q.question, "expected": ", ".join(q.gold), "top-3 retrieved": ", ".join(got)})
misses = pd.DataFrame(misses)
print(len(misses), "questions missed in the top 3"); print(misses.to_string(index=False))

## 14. Try it: retrieve rules for a SafeVision finding
This is the call the backend will make. It returns the top rules with their section number, so the report can cite them.

In [ ]:
# Try it: retrieve rules for a SafeVision finding
def find_rules(finding, k=3):
    # Top-k rule chunks for one finding: section, heading and text.
    idx = main_index.search("hybrid + re-ranker", finding, k=k)
    return [{"section": main_index.chunks.section[i], "heading": main_index.chunks.heading[i],
             "source": main_index.chunks.source[i], "text": main_index.chunks.text[i][:400]} for i in idx]

for f in ["2 of 5 workers are not wearing hard hats near an excavator.", "Worker on scaffold without a safety vest or harness."]:
    print("FINDING:", f)
    for r in find_rules(f):
        print(f"  {r['section']}  {r['heading']}")

## 15. Save the rule index for the backend, results to Excel, and ZIP

In [ ]:
# Save the rule index for the backend, results to Excel, and ZIP
import shutil
idx_dir = MODELS / "rule_index"; idx_dir.mkdir(parents=True, exist_ok=True)
main_index.chunks.to_parquet(idx_dir / "chunks.parquet")
faiss.write_index(main_index.faiss, str(idx_dir / "chunks.faiss"))
json.dump({"emb_model": EMB_MODEL, "reranker": RERANKER, "rrf_k": RRF_K, "candidates": CANDIDATES,
           "chunk_words": CHUNK_WORDS, "chunk_overlap": CHUNK_OVERLAP, "subparts": SUBPARTS}, open(idx_dir / "config.json", "w"), indent=2)

xlsx = RES / "E6_rule_search.xlsx"
with pd.ExcelWriter(xlsx) as xw:
    e6a.to_excel(xw, sheet_name="E6a_methods", index=False)
    e6b.to_excel(xw, sheet_name="E6b_chunk_size", index=False)
    if len(e6c): e6c.to_excel(xw, sheet_name="E6c_embeddings", index=False)
    misses.to_excel(xw, sheet_name="missed_questions", index=False)
    qa.assign(gold=qa.gold.str.join("; ")).to_excel(xw, sheet_name="questions", index=False)
    sections.groupby(["source", "subpart", "subpart_title"]).size().rename("sections").reset_index().to_excel(xw, sheet_name="rule_book", index=False)
    read_log().to_excel(xw, sheet_name="full_log", index=False)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb4_outputs"), "zip", OUT)
print("Saved", xlsx, "| ZIP:", archive)

In [ ]:
# [colab-only]
from google.colab import files
files.download(archive)
try:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.upload_folder(folder_path=str(OUT), repo_id=f"{api.whoami()['name']}/safevision-artifacts", path_in_repo="nb4_rule_search")
    print("Uploaded to the private HF repo.")
except Exception as e:
    print("HF upload skipped:", e)

## References

Papers used directly here are **in bold**.

9. **Zhang, Y., Wang, X., Hu, M., Zhang, Z., et al. (2025). BifrostRAG: Bridging dual knowledge graphs for multi-hop question answering in construction safety. arXiv:2507.13625; *Automation in Construction* (2026). https://doi.org/10.1016/j.autcon.2026.106794** (OSHA rule questions; hybrid search beats meaning-only search)
10. **Guo, K. X., Wong, P. K.-Y., Cheng, J. C. P., Tao, X., & Leung, P.-H. (2025). Enhancing visual-LLM through prompt engineering and hybrid retrieval-augmented generation for site safety compliance checking. *Proceedings of EC3 2025 and CIB W78*, Porto.** (hybrid keyword + meaning search for photo-based safety checks; 53.7% to 75.7%)
13. **Hou, X., Walavalkar, A., & Xiao, B. (2026). A generative AI-based construction safety assistant using retrieval-augmented generation. *Proceedings of ISARC 2026*. https://doi.org/10.22260/ISARC2026/0297** (RAG over OSHA rules, tested with 30 questions; we use the same idea of a fixed question set)
- U.S. Office of the Federal Register. *Electronic Code of Federal Regulations, Title 29 Part 1926* and eCFR API v1. https://www.ecfr.gov (public domain)
- Lewis, P., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS 33*.
- Robertson, S., & Zaragoza, H. (2009). The probabilistic relevance framework: BM25 and beyond. *Foundations and Trends in Information Retrieval*, 3(4), 333-389.
- Cormack, G. V., Clarke, C. L. A., & Büttcher, S. (2009). Reciprocal rank fusion outperforms Condorcet and individual rank learning methods. *SIGIR 2009*, 758-759.
- Xiao, S., Liu, Z., Zhang, P., & Muennighoff, N. (2023). C-Pack: Packaged resources to advance general Chinese embedding (BGE models and re-ranker). arXiv:2309.07597.
- Johnson, J., Douze, M., & Jégou, H. (2019). Billion-scale similarity search with GPUs (FAISS). *IEEE Transactions on Big Data*, 7(3), 535-547.